# Finance Advisor: No Memory vs. With Memory

A personal finance advisor agent built with the OpenAI Agents SDK. Below we compare two runs of the same two queries:
- **Task 1 & 2 (no memory):** each query is independent — no session attached.
- **Task 3 (with memory):** a `SQLiteSession` is attached so the agent remembers the expenses from query 1 when answering query 2.

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner

load_dotenv()

True

In [2]:
finance_advisor = Agent(
    name="Finance Advisor",
    model="gpt-5.4-mini",
    instructions=(
        "You are a personal finance advisor. Help the user track spending by category "
        "(e.g., groceries, transportation, dining, entertainment) and give practical, "
        "specific budget advice. When the user reports expenses, summarize the totals "
        "by category. When asked for advice, suggest concrete areas to cut back and "
        "explain your reasoning. Be concise and actionable."
    ),
)

## Tasks 1 & 2 — No Memory

Each `Runner.run` call is independent. The agent will not recall the expenses from query 1 when answering query 2.

In [3]:
query1 = "I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week."
result1 = await Runner.run(finance_advisor, query1)
print(result1.final_output)

Here’s your spending summary for the week:

- Groceries: $120
- Transportation (Uber): $40
- Dining (restaurants): $60

Total spent: $220

Quick take:
- Groceries are your biggest category, which is usually a good place to focus on efficiency.
- Restaurants are a flexible area to cut back if you want to lower spending fast.
- Uber is moderate, but could be reduced by combining trips or using transit when practical.

If you want, I can also help you set a weekly budget target for these categories.


In [4]:
query2 = "My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?"
result2 = await Runner.run(finance_advisor, query2)
print(result2.final_output)

I don’t have any of your past spending details in this chat yet, so I can’t identify the best cuts “based on everything you told me so far.”

If you send me your recent spending by category, I can tell you exactly where to cut. For example, share something like:

- Groceries: $__
- Dining out: $__
- Transportation: $__
- Entertainment: $__
- Shopping/misc: $__

For a $250 weekly budget, the fastest places to cut are usually:
- Dining out/coffee: set a hard cap or swap 2–3 meals to home-cooked
- Entertainment: pause paid outings/subscriptions for the week
- Shopping/misc: put a zero-dollar limit unless it’s essential
- Transportation: combine trips and avoid ride-shares if possible

If you want, send your last week’s spending and I’ll give you a specific cut plan for next week.


## Task 3 — With Memory (SQLiteSession)

Now we attach a `SQLiteSession` to both runs. The session persists conversation history, so the second query has full context of the expenses reported in the first.

In [5]:
from agents import SQLiteSession

session = SQLiteSession("finance_advisor_demo")

result1_mem = await Runner.run(finance_advisor, query1, session=session)
print("--- Query 1 (with memory) ---")
print(result1_mem.final_output)

--- Query 1 (with memory) ---
Here’s your spending summary for this week:

- Groceries: $120
- Transportation (Uber): $40
- Dining (restaurants): $60

Total spent: $220

Quick take:
- Groceries are your biggest category, which is usually okay if most meals are home-cooked.
- Dining is fairly significant at $60; this is an easy place to trim if you want to save.
- Uber at $40 is moderate, but if it’s recurring, consider whether some trips could be replaced with transit, walking, or batching errands.

If you want to cut back next week, the easiest targets are:
1. Restaurants — set a cap, like $30–$40.
2. Uber — limit to essential trips only.
3. Groceries — focus on a meal plan and a short shopping list to avoid impulse buys.

If you want, I can also help you turn this into a weekly budget target.


In [6]:
result2_mem = await Runner.run(finance_advisor, query2, session=session)
print("--- Query 2 (with memory) ---")
print(result2_mem.final_output)

--- Query 2 (with memory) ---
You’ve spent **$220 of your $250 weekly budget**, so you have **$30 left**.

Best places to cut next week:
- **Restaurants**: easiest cut. If you keep dining out to **$20–$30**, you’ll free up room fast.
- **Uber**: try to keep it to **$0–$20** if possible by using transit, walking, or combining trips.
- **Groceries**: this is worth keeping, but you may want to aim for **$100–$110** by planning meals and avoiding extras.

A practical target for next week:
- Groceries: **$110**
- Uber: **$20**
- Restaurants: **$20**
- Total: **$150**

That would put you **$100 under budget**.

If you want, I can help you build a simple weekly spending plan that fits your $250 budget.


### Side-by-side comparison

- **Without memory (Task 2):** the agent has no record of the $120 / $40 / $60 expenses, so the advice is generic.
- **With memory (Task 3):** the agent recalls the reported expenses ($220 total against a $250 budget) and gives concrete cuts grounded in those numbers.